# SatQuery: prepare the BigEarthNet.txt benchmark images (CPU only, no GPU quota)

Downloads and renders only the images of the benchmark split that `satquery_vlm.ipynb` scores
(200 examples per question type, same seed, so the same 800 examples). Run it once; the scoring notebook
`satquery_vlm_score.ipynb` then uses its output, so a GPU session no longer spends hours downloading.

**Run:** right panel → **Accelerator: None**, **Internet: On** → **Save Version → Save & Run All (Commit)**.
The download streams the 118 GB archives from Zenodo and keeps only the needed patches, so it takes hours
even though the result is small (about 1,600 PNGs). It stops after 11 hours and keeps what it has.

In [ ]:
import os, time, subprocess
START = time.time()
REPO, OUT = "/tmp/SatQuery", "/kaggle/working/bench"
BENCH_PER_TYPE = 200     # must match satquery_vlm.ipynb
ARGS = f"--out {OUT} --splits bench --bench-per-type {BENCH_PER_TYPE}"

In [ ]:
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/Vansh-Vaid/SatQuery.git", REPO], check=True)
os.chdir(REPO)
!pip install -q "transformers>=4.57" peft accelerate zstandard rasterio opencv-python-headless reportlab python-multipart
# Kaggle ships torchao 0.10; current peft refuses any torchao older than 0.16. SatQuery does not use it.
!pip uninstall -y -q torchao
!git log --oneline -1

In [ ]:
budget = int(11 * 60 - (time.time() - START) / 60 - 20)
print(f"downloading for at most {budget} minutes")
!python training/fetch_bigearthnet_subset.py {ARGS} --time-limit-min {budget}
!python training/prepare_bigearthnet_txt.py --rendered {ARGS}
!wc -l {OUT}/bench.jsonl
print(f"Done in {(time.time() - START) / 3600:.1f} h. Output: {OUT} (bench.jsonl + images/)")